In dit notebook laden we de data in, passen we de gekozen baseline-correctie (Hybrid 4S+ASL) en normalisatie (SNV) methode toe. Vervolgens exporteren we de bewerkte data naar een nieuw CSV-bestand. Hier is voor gekozen, omdat het toepassen van de baseline-correctie op alle spectra alleen al ongeveer 90 minuten duurt. Door de bewerkte data op te slaan, hoeven we deze stap niet telkens opnieuw uit te voeren.

In [1]:
import os
import sys
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
# Add the parent directory (project root) to Python path
project_root = os.path.abspath('..')  # Go up one level from current notebook
if project_root not in sys.path:
    sys.path.insert(0, project_root)


from utils.baseline_correction_functions import BaselineCorrector
from utils.libs_generic_functions import LibsDataLoader, LibsDataPreprocessor
from utils.normalization_functions import Normalizer
from utils.saturation_functions import apply_voigt_saturation_correction

In [2]:
dataloader = LibsDataLoader(data_directory='data')
preprocessor = LibsDataPreprocessor()
corrector = BaselineCorrector()

In [3]:
df = dataloader.load_all_measurements_no_rounding()

MEMORY-EFFICIENT LOADING OF ALL INDIVIDUAL MEASUREMENTS
First pass: Counting measurements...
Processing file 1/316: 2024-11-05T11-26-41_nr-001_B1_tread_aided_10Hz_280A.h5
Processing file 11/316: 2024-11-05T11-51-23_nr-011_B4_innerliner_aided_10Hz_280A.h5
Processing file 21/316: 2024-11-05T12-02-34_nr-021_B7_sidewall_aided_10Hz_280A.h5
Processing file 31/316: 2024-11-05T12-13-30_nr-031_B22_tread_aided_10Hz_280A.h5
Processing file 41/316: 2024-11-05T12-25-42_nr-041_B33_innerliner_aided_10Hz_280A.h5
Processing file 51/316: 2024-11-05T12-36-31_nr-051_B28_sidewall_aided_10Hz_280A.h5
Processing file 61/316: 2024-11-05T12-47-29_nr-061_B15_tread_aided_10Hz_280A.h5
  Counted 10,000 measurements...
Processing file 71/316: 2024-11-05T12-58-45_nr-071_B30_innerliner_aided_10Hz_280A.h5
Processing file 81/316: 2024-11-05T13-17-11_nr-081_B35_innerliner_aided_10Hz_280A.h5
Processing file 91/316: 2024-11-05T14-05-16_nr-091_B26_sidewall_aided_10Hz_280A.h5
Processing file 101/316: 2024-11-05T14-16-52_nr-1

In [4]:
df.head()

,200.000,200.098,200.195,200.293,200.391,200.489,200.586,200.684,200.782,200.879,...,999.414,999.511,999.609,999.707,999.805,999.902,1000.000,origin,tire_number,measurement_id
0,201.0,156.0,249.0,195.0,271.0,142.0,240.0,232.0,225.0,165.0,...,480.0,524.0,480.0,512.0,452.0,534.0,458.0,tread,1,0
1,166.0,124.0,208.0,104.0,209.0,91.0,232.0,168.0,202.0,138.0,...,545.0,550.0,524.0,623.0,529.0,606.0,503.0,tread,1,1
2,200.0,181.0,282.0,183.0,276.0,158.0,224.0,187.0,221.0,176.0,...,486.0,520.0,458.0,536.0,423.0,549.0,422.0,tread,1,2
3,143.0,113.0,160.0,118.0,203.0,71.0,200.0,130.0,195.0,145.0,...,536.0,575.0,532.0,651.0,525.0,624.0,527.0,tread,1,3
4,235.0,162.0,237.0,196.0,255.0,147.0,301.0,266.0,254.0,225.0,...,484.0,541.0,479.0,583.0,484.0,556.0,472.0,tread,1,4


In [5]:
non_feature_cols = ['tire_number', 'origin', 'measurement_id']
print(f"Non-feature columns: {non_feature_cols}")

# Get numeric columns (wavelength data)
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
wavelength_columns = [col for col in numeric_columns if col not in non_feature_cols]

Non-feature columns: ['tire_number', 'origin', 'measurement_id']


In [6]:
df = apply_voigt_saturation_correction(df, saturation_threshold=65535, verbose=True)


🔧 Voigt Profile Saturation Correction
Total measurements: 51453
Saturated measurements: 14224
Processing measurements: 14224
Saturation threshold: 65535
Minimum peak width: 3
Progress: 1422/14224 (10.0%)
Progress: 2844/14224 (20.0%)
Progress: 4266/14224 (30.0%)
Progress: 5688/14224 (40.0%)
Progress: 7110/14224 (50.0%)
Progress: 8532/14224 (60.0%)
Progress: 9954/14224 (70.0%)
Progress: 11376/14224 (80.0%)
Progress: 12798/14224 (90.0%)
Progress: 14220/14224 (100.0%)

📊 Correction Summary:
------------------------------
Measurements processed: 14224
Measurements corrected: 11325
Success rate: 79.6%
Total peaks found: 20942
Total peaks corrected: 20915
Peak correction rate: 99.9%

Intensity ranges:
Original max: 65,535
Corrected max: 1,287,537
Max increase: 19.65x


268 minunten

In [9]:
df = dataloader.round_wavelengths(df)

Rounding wavelength columns and removing duplicates...
Original wavelength columns: 8188
After rounding: 8001 unique wavelengths
Creating DataFrame with deduplicated wavelength columns...
  Averaged 2 columns into 202.2 nm
  Averaged 2 columns into 206.4 nm
  Averaged 2 columns into 210.7 nm
  Averaged 2 columns into 215.0 nm
  Averaged 2 columns into 219.2 nm
  Averaged 2 columns into 223.6 nm
  Averaged 2 columns into 227.8 nm
  Averaged 2 columns into 232.1 nm
  Averaged 2 columns into 236.4 nm
  Averaged 2 columns into 240.6 nm
  Averaged 2 columns into 244.9 nm
  Averaged 2 columns into 249.2 nm
  Averaged 2 columns into 253.5 nm
  Averaged 2 columns into 257.8 nm
  Averaged 2 columns into 262.0 nm
  Averaged 2 columns into 266.3 nm
  Averaged 2 columns into 270.6 nm
  Averaged 2 columns into 274.8 nm
  Averaged 2 columns into 279.2 nm
  Averaged 2 columns into 283.4 nm
  Averaged 2 columns into 287.7 nm
  Averaged 2 columns into 292.0 nm
  Averaged 2 columns into 296.2 nm
  Avera

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51453 entries, 0 to 51452
Columns: 8004 entries, tire_number to 1000.0
dtypes: float32(8001), int32(2), object(1)
memory usage: 1.5+ GB


In [12]:
df.head()

,tire_number,origin,measurement_id,200.0,200.1,200.2,200.3,200.4,200.5,200.6,...,999.1,999.2,999.3,999.4,999.5,999.6,999.7,999.8,999.9,1000.0
0,1,tread,0,201.0,156.0,249.0,195.0,271.0,142.0,240.0,...,533.0,497.0,549.0,480.0,524.0,480.0,512.0,452.0,534.0,458.0
1,1,tread,1,166.0,124.0,208.0,104.0,209.0,91.0,232.0,...,601.0,550.0,606.0,545.0,550.0,524.0,623.0,529.0,606.0,503.0
2,1,tread,2,200.0,181.0,282.0,183.0,276.0,158.0,224.0,...,533.0,472.0,535.0,486.0,520.0,458.0,536.0,423.0,549.0,422.0
3,1,tread,3,143.0,113.0,160.0,118.0,203.0,71.0,200.0,...,584.0,522.0,603.0,536.0,575.0,532.0,651.0,525.0,624.0,527.0
4,1,tread,4,235.0,162.0,237.0,196.0,255.0,147.0,301.0,...,570.0,491.0,529.0,484.0,541.0,479.0,583.0,484.0,556.0,472.0


In [14]:
wavelength_columns

['200.000',
 '200.098',
 '200.195',
 '200.293',
 '200.391',
 '200.489',
 '200.586',
 '200.684',
 '200.782',
 '200.879',
 '200.977',
 '201.075',
 '201.173',
 '201.270',
 '201.368',
 '201.466',
 '201.563',
 '201.661',
 '201.759',
 '201.857',
 '201.954',
 '202.052',
 '202.150',
 '202.247',
 '202.345',
 '202.443',
 '202.541',
 '202.638',
 '202.736',
 '202.834',
 '202.931',
 '203.029',
 '203.127',
 '203.225',
 '203.322',
 '203.420',
 '203.518',
 '203.615',
 '203.713',
 '203.811',
 '203.909',
 '204.006',
 '204.104',
 '204.202',
 '204.299',
 '204.397',
 '204.495',
 '204.593',
 '204.690',
 '204.788',
 '204.886',
 '204.984',
 '205.081',
 '205.179',
 '205.277',
 '205.374',
 '205.472',
 '205.570',
 '205.668',
 '205.765',
 '205.863',
 '205.961',
 '206.058',
 '206.156',
 '206.254',
 '206.352',
 '206.449',
 '206.547',
 '206.645',
 '206.742',
 '206.840',
 '206.938',
 '207.036',
 '207.133',
 '207.231',
 '207.329',
 '207.426',
 '207.524',
 '207.622',
 '207.720',
 '207.817',
 '207.915',
 '208.013',
 '20

In [15]:
# Update wavelength columns after rounding
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
wavelength_columns = [col for col in numeric_columns if col not in non_feature_cols]

X = df[wavelength_columns].values
y = df['origin'].values

# Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Dataset shape: {X.shape}")
print(f"Classes: {label_encoder.classes_}")
print(f"Class distribution: {np.bincount(y_encoded)}")

X_corrected = corrector.apply_correction_batch(X)

# 2. Standardize features
normalizer = Normalizer() 
X_scaled = normalizer.apply_standard_normal_variate_on_dataset(X_corrected)

Dataset shape: (51453, 8001)
Classes: ['innerliner' 'sidewall' 'tread']
Class distribution: [17260 16257 17936]
Applying HYBRID baseline correction to 51453 spectra in batches of 100...
  Processing batch 1/515 (spectra 1-100)
  Processing batch 2/515 (spectra 101-200)
  Processing batch 3/515 (spectra 201-300)
  Processing batch 4/515 (spectra 301-400)
  Processing batch 5/515 (spectra 401-500)
  Processing batch 6/515 (spectra 501-600)
  Processing batch 7/515 (spectra 601-700)
  Processing batch 8/515 (spectra 701-800)
  Processing batch 9/515 (spectra 801-900)
  Processing batch 10/515 (spectra 901-1000)
  Processing batch 11/515 (spectra 1001-1100)
  Processing batch 12/515 (spectra 1101-1200)
  Processing batch 13/515 (spectra 1201-1300)
  Processing batch 14/515 (spectra 1301-1400)
  Processing batch 15/515 (spectra 1401-1500)
  Processing batch 16/515 (spectra 1501-1600)
  Processing batch 17/515 (spectra 1601-1700)
  Processing batch 18/515 (spectra 1701-1800)
  Processing bat

81 minuten

In [16]:
#replace the original data with the processed data
df[wavelength_columns] = X_scaled
print("✅ DataFrame updated with corrected and normalized data")



✅ DataFrame updated with corrected and normalized data


In [17]:
processed_data_path = 'data/csv/processed_data.csv'
df.to_csv(processed_data_path, index=False)

In [ ]:
# Verify by loading the CSV
import pandas as pd
df = pd.read_csv(processed_data_path)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51453 entries, 0 to 51452
Columns: 6702 entries, tire_number to 982.4
dtypes: float64(6699), int64(2), object(1)
memory usage: 2.6+ GB
